In [ ]:
#The code in this setup is designed to create the plots that map the sidelobes of the radio antennas using
#the information gleaned from pass observed in the stow position.

#azim and elev are lists of the predicted azimuth and elevations as predicted by TLE sets
#rbinnumber,abinnumber should be set to 7 and 33 respectively
#Title String is the title of the plot
#Mode details whether the graph will display the mean, median, standard deviation or difference between two datasets
#SubtractedThresholdedAmpValues is the list of peak antenna gain values in each plot 

import matplotlib.pyplot as plt
import numpy as np
import skyfield
from skyfield.api import load, wgs84, EarthSatellite
import astropy.coordinates
def SideLobesMap01(azim,elev,SubtractedThresholdedAmpValues,rbinnumber,abinnumber,titlestring,mode):
    import matplotlib.pyplot as plt
    ListOfIntensitiesByAngleBinMatrix = [[[[0] for z in range(0)] for x in range((rbinnumber))] for y in range((abinnumber))]
    MedianOfIntensitiesByAngleBinMatrix = [[[[0] for z in range(0)] for x in range((rbinnumber))] for y in range((abinnumber))]
    MeanOfIntensitiesByAngleBinMatrix = [[[[0] for z in range(0)] for x in range((rbinnumber))] for y in range((abinnumber))]
    StdDevOfIntensitiesByAngleBinMatrix = [[[[0] for z in range(0)] for x in range((rbinnumber))] for y in range((abinnumber))]
    #print(MedianOfIntensitiesByAngleBinMatrix)
    rbins = np.linspace(0, 90, rbinnumber)
    abins = np.linspace(0, 360* np.pi/180, abinnumber)       

    for i in range(len(SubtractedThresholdedAmpValues)):
        x_value = roundup_a(azim[i],abinnumber)#; print(x_value,pol017hbPass1ThresholdedAzimuth[i])
        y_value = roundup_r(elev[i],rbinnumber)#; print(y_value,pol017hbPass1ThresholdedAltitude[i])
        ListOfIntensitiesByAngleBinMatrix[x_value][y_value].append(SubtractedThresholdedAmpValues[i])
    #ListOfIntensitiesByAngleBinMatrix
    for i in range(len(MedianOfIntensitiesByAngleBinMatrix)):
        for j in range(len(MedianOfIntensitiesByAngleBinMatrix[0])):
            if len(ListOfIntensitiesByAngleBinMatrix[i][j])>0:
                MedianOfIntensitiesByAngleBinMatrix[i][j].append(np.median(ListOfIntensitiesByAngleBinMatrix[i][j]))
                MeanOfIntensitiesByAngleBinMatrix[i][j].append(np.mean(ListOfIntensitiesByAngleBinMatrix[i][j]))
                StdDevOfIntensitiesByAngleBinMatrix[i][j].append(np.std(ListOfIntensitiesByAngleBinMatrix[i][j],ddof=1))

    
    
    bin_selection_r = []
    bin_selection_a = []
    MedianValues1Dlist = []
    MeanValues1Dlist = []
    StdDevValues1Dlist = []

    for i in range(len(MedianOfIntensitiesByAngleBinMatrix)):
        for j in range(len(MedianOfIntensitiesByAngleBinMatrix[0])):
            if len(MedianOfIntensitiesByAngleBinMatrix[i][j])>0:
                bin_selection_r.append(j*90/rbinnumber)
                bin_selection_a.append(i*360*np.pi/(180*abinnumber))
                MedianValues1Dlist.append(MedianOfIntensitiesByAngleBinMatrix[i][j][0])
                MeanValues1Dlist.append(MeanOfIntensitiesByAngleBinMatrix[i][j][0])
                StdDevValues1Dlist.append(StdDevOfIntensitiesByAngleBinMatrix[i][j][0])
        
    Difference = []            
    for i in range(len(MedianValues1Dlist)):
        Difference.append(MedianValues1Dlist[i]-MeanValues1Dlist[i])
    
    if mode == "Median":
        hist, _, _ = np.histogram2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=MedianValues1Dlist)#,cmin = 1)
    if mode == "Mean":
        hist, _, _ = np.histogram2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=MeanValues1Dlist)#,cmin = 1)
    if mode == "Standard Deviation":
        hist, _, _ = np.histogram2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=StdDevValues1Dlist)#,cmin = 1)
    if mode == "Difference":
        hist, _, _ = np.histogram2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=Difference)#,cmin = 1)
    #hist, _, _ = plt.hist2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=MedianValues1Dlist,cmin = 1)
    A, R = np.meshgrid(abins, rbins)

    # plot
    fig, ax = plt.subplots(subplot_kw=dict(projection="polar"))#,figsize=(7, 7))#,cmin = 1
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    #import matplotlib as mpl
    #cmap1 = mpl.cm.get_cmap('magma').copy()  # viridis is the default colormap for imshow
    #cmap.set_bad(color='red')
    #cmap1.set_bad(color='white')

    #my_cmap = mpl.cm.get_cmap("magma").copy()
    #my_cmap.set_under('white',1)
    from matplotlib.colors import ListedColormap
    base_cmap = plt.cm.magma
    colors = base_cmap(np.linspace(0, 1, 256))
    colors[0] = [1, 1, 1, 1]  # RGBA for white
    custom_cmap = ListedColormap(colors)
    
    if mode == "Standard Deviation" or "Difference":
        pc = ax.pcolormesh(A, R, hist.T, cmap=custom_cmap)#my_cmap)#cmap=cmap1)
    else:
        pc = ax.pcolormesh(A, R, hist.T, cmap=custom_cmap, vmin=0, vmax=15)#my_cmap)#cmap=cmap1)
    #print(hist.T)
    cbar = fig.colorbar(pc)
    cbar.set_label('Antenna Gain [dB]', rotation=270, labelpad=15)
    ax.set_title(titlestring)
    ax.grid(True)
    ax.set_rlim(bottom=90, top=0)
    ax.tick_params(axis='y', colors='black')
    plt.show()